# Retrieval-Augmented Generation (RAG) with LlamaIndex


**Retrieval-Augmented Generation (RAG)** is a hot topic since it gives LLMs access to information that has not been in their training data. RAG...

- ...combines information retrieval with generative models to enhance response quality
- ... pulls data from extensive databases to inform responses.
- ... reduces the likelihood of generating plausible but incorrect or unfounded answers by directly incorporating factual data

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/rag_overview.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    Retrieval-augmented generation: the retriever finds relevant passages, the generator answers with them
  </div>
</div>

## Example: Chat with your Documents 

In the following we are going to build a simple chatbot that can answer user queries from a document database.

In [ ]:
from pathlib import Path

from ai_dojo import show

Suppose we want to build a chatbot to help us study a topic we are intested in - artificial intelligence. Wikipedia articles are a great resource, but they can be long reads. What if we could ask specific questions to be answered from the documents?

As a starting point we pick three articles:

- [Artificial Intelligence](https://en.wikipedia.org/wiki/Artificial_intelligence)
- [Deep Learning](https://en.wikipedia.org/wiki/Deep_learning)
- [Large Language Models](https://en.wikipedia.org/wiki/Large_language_model)

### Document Preparation

The first step is usually to convert the original documents into a pure text format that an LLM can operate on. 
- We want text, but also to preserve as much information and structure of the document.
- A popular format for this purpose is _Markdown_.
- Depending on the document format, structure and content, the proper conversion can be a challenging part of the project.


For our Wikipedia knowledge base, we decide to start from the Wiki source text (Mediawiki syntax). This is already a pure text format and could be suitable for our RAG system. 

In [ ]:
document_dir = Path("../.assets/data/wikipedia-ai/")


In [ ]:
mediawiki_document_dir = document_dir / "mediawiki"

In [ ]:
ls {mediawiki_document_dir}

In [ ]:
!head {mediawiki_document_dir}/wikipedia-ai.mediawiki 

Nevertheless, we convert it into the more readable and more common Markdown. A convenient tool for this is `pandoc`:

In [ ]:
show.command("pandoc -f mediawiki -t markdown_strict -o md/wikipedia-llm.md mediawiki/wikipedia-llm.mediawiki")

In [ ]:
markdown_document_dir = document_dir / "md"

In [ ]:
ls {markdown_document_dir}

In [ ]:
!head {markdown_document_dir}/wikipedia-ai.md

In [ ]:
with open(markdown_document_dir / "wikipedia-ai.md") as md_file:
    show.text(md_file.read(), limit=4000)

### Building a Query Engine with `llama_index`

After we brought our documents into the right textual format, we are ready to start building a RAG system.

`llama_index` gives us a wide range of useful tools for this and similar applications. Here are a few components we need:

**Setting ollama URL**

In [ ]:
from ai_dojo.lm import get_ollama_url

ollama_url = get_ollama_url()
ollama_url

**Managing Settings**

The `Settings` object manages defaults and custom settings that apply to our RAG system.

In [ ]:
from llama_index.core import Settings


The default language model that generates responses is set to `OpenAI`, expecting to query the proprietary API of OpenAI. 

In [ ]:
try:
    Settings.llm
except Exception as ex:
    print(ex)

Rather, we want to build the RAG application on our own local models, served via _Ollama_.

**Selecting the Chat Model**

The chat model receives our query and the retrieved document parts in order to generate an answer. When chosing a model, we are looking for a balance between model size and general question answering capabilities.

In [ ]:
import ollama

In [ ]:
chat_model_name = "qwen3.5:9b"

... make it available via `llama_index`...

In [ ]:
from llama_index.llms.ollama import Ollama

In [ ]:
chat_model = Ollama(
    model=chat_model_name,
    base_url=ollama_url,
    request_timeout=90,
)
chat_model

... and set it as the default chat model for our application:

In [ ]:
Settings.llm = chat_model

**Selecting the Embedding Model**

In addition to the chat model, we need another LLM type in our RAG system: A **text embedding model** maps chunks of text to vectors in a vector space. 
- The model has been trained so that the vector representation of the text captures the meaning of the text. 
- Two text segments with similar content should have a low distance in the embedding space. 
- This is what enables **semantic similarity search**: 
  1. the embedding model generates an embedding vector for the query
  2. previously, segments ("chunks") of the documents have been embedded with the same model
  3. the embedding space is searched for vectors that are close to the embedding vector of the query
  4. the corresponding document chunks are retrieved and passed to the context of the chat model


The _Ollama_ model registry contains several specialized embedding models. These are relatively small compared to chat models.

In [ ]:
#embed_model_name = "mxbai-embed-large:latest"
embed_model_name = "nomic-embed-text:latest"

We ensure that the model is availabe in our Ollama instance...

In [ ]:
response = ollama.pull(embed_model_name)
print(response)
assert response["status"] == "success"

... make it available via `llama_index`...

In [ ]:
from llama_index.embeddings.ollama import OllamaEmbedding


In [ ]:
embed_model_remote_name = "nomic-embed-text:latest"

embed_model = OllamaEmbedding(
    model_name=embed_model_name,
    base_url=ollama_url,
)
embed_model

... and set it as the default embedding model for our application:

In [ ]:
Settings.embed_model = embed_model


**Loading and Splitting the Documents**

In [ ]:
from llama_index.core import SimpleDirectoryReader

In [ ]:
doc_file_paths = list(markdown_document_dir.glob('*.md'))
doc_file_paths  

In [ ]:
documents = SimpleDirectoryReader(
    input_files=doc_file_paths,
).load_data()

In [ ]:
len(documents)

In [ ]:
for i in range(3):
    show.text(documents[i].text)
    show.text("----------------")


**Building the Vector Database**

Another central component of a RAG system is the **vector database** (or **vector store**):
- In a vector database, data records are stored with a vector as their key.
- Vector databases typically implement Approximate Nearest Neighbor (ANN) algorithms, so that one can search the database with a query vector to retrieve the closest matching database records.

<div style="text-align: center; margin: 30px 0;">
  <img src="graphics/vector_db.svg" style="max-width: 800px; max-height: 600px; border: 2px solid #ddd; border-radius: 8px; padding: 5px; box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
  <div style="margin-top: 10px; font-style: italic; color: #666; font-size: 0.9em;">
    How a vector database is written and queried
  </div>
</div>

`llama_index` makes it straightforward to place document chunks in a vector database for RAG. 

In [ ]:
from llama_index.core import VectorStoreIndex

**Set chunk size and overlap between chunks**

In [ ]:
# we can define the chunk size and overlap between the chunks
Settings.chunk_size = 512
Settings.chunk_overlap = 50

**Build vector index**

In [ ]:
index = VectorStoreIndex.from_documents(
    documents=documents,
)
index

**Building the Query Engine**


In [ ]:
query_engine = index.as_query_engine(
    streaming=True, # response as stream?
)
query_engine

**First Test of the Query Engine**

In [ ]:
response = query_engine.query(
    "When was the first neural network proposed?"
)
response.print_response_stream()

In [ ]:
response = query_engine.query(
    "Define 'GOFAI'."
)
response.print_response_stream()

In [ ]:
response = query_engine.query(
    "How many parameters does GPT-4 have?"
)
response.print_response_stream()

In [ ]:
# your query here

**Inspecting the RAG Responses**

Interested in which parts of the document were used as context for the response? We can ask a response object for the document chunks (called "nodes" in the database) that were used.

In [ ]:
response = query_engine.query(
    "What did Allan Turing write about defining artificial intelligence?"
)
response.print_response_stream()

In [ ]:
response.get_formatted_sources()

In [ ]:
response = query_engine.query(
    "Where and when did the workshop take place that founded the field of AI resarch?"
)
response.print_response_stream()

In [ ]:
response.get_formatted_sources()

**Building a Chat Engine**

A query engine answers each query separately, but it maintains no state, you cannot reference earlier queries and responses. If you want chat-like behavior in a RAG system, you need to experiment with the `llama_index`**chat engine**.

In [ ]:
chat_engine = index.as_chat_engine()


In [ ]:
response = chat_engine.stream_chat(
    "When was the first neural network proposed?"
)
response.print_response_stream()

In [ ]:
response = chat_engine.stream_chat(
    "What was it able to do?"
)
response.print_response_stream()

In [ ]:
response = chat_engine.stream_chat(
    "Are similar models used nowadays?"
)
response.print_response_stream()

## Tweaking the RAG System

There are many parameters in a RAG system that could be customized in pursuit of better answers. We introduce a few basic ones.

**Document Chunk Size and Overlap**

_How does the way we split the document(s) into segments influence the response quality?_

In [ ]:
from llama_index.core.node_parser import SentenceSplitter


In [ ]:
splitter = SentenceSplitter.from_defaults(
    chunk_size=100, # in tokens
    chunk_overlap=20,
)

In [ ]:
nodes = splitter.get_nodes_from_documents(documents=documents)
nodes[:3]

<div style="border:2px solid gray; padding:10px; width: 95%;">
<b>🫳 Hands-on: Experiment with the document splitting parameters</b>

_Can you get improved answers by tuning these parameters?_

## Retrieval Augmented Generation: Bring Your Own Documents

<div style="border:2px solid gray; padding:10px; width: 95%;">
<b>🫳 Hands-on: Build a RAG query engine for your own documents</b>

Can you think of a long document or set of documents that you would like to ask questions from?  Try the workflow described above on your own documents!

In [ ]:
# your code here

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_